# CECL-style Expected Loss Model: Freddie Mac Mortgage PD, Validation and Stress Test

**Data:** Freddie Mac Single-Family Loan-Level Dataset, 2018 vintage sample (50,000 US mortgages; one origination file and one monthly performance file). It is free after registering on Freddie Mac's Clarity site and is licensed for internal use, so the raw files are not included with this notebook.

**Goal:** estimate each loan's probability of default (PD) over its first 24 months, check how well the model separates defaulters from non-defaulters and whether its probabilities are accurate, turn PD into a 24-month expected credit loss (ECL), and see how that loss changes under a Federal Reserve severely adverse house-price shock.

**How this relates to CECL:** CECL (Current Expected Credit Loss, the US accounting standard) requires lenders to reserve for expected losses over the life of a loan, using reasonable and supportable forecasts. This notebook builds the same ingredients (PD, LGD, EAD and a macro scenario) on a 24-month window, so it is a CECL-*style* framework and not a full CECL model. The limitations are listed at the end.

**Key terms**

- **PD** (probability of default): the chance a borrower defaults in the window.
- **LGD** (loss given default): the share of the balance lost if the borrower defaults.
- **EAD** (exposure at default): the amount owed when default happens.
- **ECL** (expected credit loss) = PD x LGD x EAD.
- **LTV** (loan-to-value): loan amount divided by property value. **DTI** (debt-to-income): monthly debt payments divided by monthly income.

**Note:** US mortgage data used to demonstrate the credit-risk modelling method: model development, validation and a stress overlay.

## 0. Look at the raw files first

- The Freddie Mac files are plain text with columns separated by `|` and **no header row**, so until we name the columns we only have column numbers.
- `nrows=5` reads just five rows. The performance file has about two million rows, so peeking first is fast and safe.
- The **origination file** has one row per loan (the borrower and the loan terms at the start). The **performance file** has one row per loan per month (balance and delinquency status).
- Both files carry the same loan ID (column 19 in the origination file, column 0 in the performance file). That ID is how the two are joined later.

In [11]:
import pandas as pd

orig = pd.read_csv("sample_orig_2018.txt", sep="|", header=None, nrows=5)
perf = pd.read_csv("sample_perf_2018.txt", sep="|", header=None, nrows=5)

print(orig.shape, perf.shape)
print(orig.to_string())
print(perf.to_string())

(5, 31) (5, 35)
    0       1  2       3        4   5   6  7   8   9       10  11    12 13 14   15  16  17   18            19 20   21  22     23 24  25   26 27  28 29    30
0  654  201803  N  204802      NaN   0   1  P  77  35   50000  77  4.50  R  N  FRM  KY  SF  421  F18Q10000028  P  360   1  OTHER  N NaN  NaN  N   7  N  9999
1  693  201803  N  203302  24340.0   0   1  P  80  41  132000  80  3.25  R  N  FRM  MI  SF  493  F18Q10000052  P  180   2  OTHER  N NaN  NaN  N   1  N  9999
2  757  201803  Y  204802      NaN  25   1  P  97  10   28000  97  4.00  R  N  FRM  IA  SF  513  F18Q10000084  P  360   1  OTHER  N NaN    H  N   7  N  9999
3  807  201803  N  203302      NaN   0   1  P  26  38   52000  26  3.25  R  N  FRM  IA  SF  505  F18Q10000117  C  180   2  OTHER  N NaN  NaN  N   7  N  9999
4  812  201803  N  203302  19340.0   0   1  P  14  14   35000  14  3.75  R  N  FRM  IA  SF  528  F18Q10000140  P  180   2  OTHER  N NaN  NaN  N   1  N  9999
             0       1        2   3   4   

**Reading the output:** the origination file has 31 columns and the performance file has 35. The first loan (ID `F18Q10000028`) appears once in the origination file and in many monthly rows of the performance file, with the loan age rising 0, 1, 2, 3, 4.

## 1. Load the data and name the columns we need

- Only the columns used in the model are loaded (`usecols`), which keeps the large performance file light on memory.
- Column positions were matched to the actual values in the first rows (for example, column 0 holds scores like 654 and 693, and column 12 holds rates like 4.50). They should be cross-checked against Freddie Mac's file layout document.
- The delinquency status is read as text (`dtype={3: str}`) because it can hold a code such as `RA` as well as numbers.

**Origination columns kept**

| Column | Name | Meaning |
|---|---|---|
| 0 | credit_score | Borrower credit score at origination (9999 = not available) |
| 1 | first_pay_date | First payment date (YYYYMM) |
| 2 | first_time_buyer | First-time home buyer, Y or N |
| 7 | occupancy | P = principal residence, S = second home, I = investment property |
| 8 | cltv | Combined loan-to-value, % (999 = not available) |
| 9 | dti | Debt-to-income ratio, % (999 = not available) |
| 10 | orig_upb | Original loan balance, US dollars |
| 11 | ltv | Loan-to-value, % (999 = not available) |
| 12 | int_rate | Interest rate on the loan, % |
| 16 | state | Property state |
| 19 | loan_id | Unique loan identifier, used to join the two files |
| 20 | purpose | P = purchase, C = cash-out refinance, N = no cash-out refinance |
| 21 | term | Loan term in months |
| 22 | num_borrowers | Number of borrowers |

**Performance columns kept:** `loan_id`, `month` (reporting month, YYYYMM), `upb` (current balance), `dq_status` (delinquency status) and `loan_age` (months since the loan started).

In [12]:
import pandas as pd
import numpy as np

# Origination file: keep only the columns we need
orig_cols = {0:"credit_score", 1:"first_pay_date", 2:"first_time_buyer",
             7:"occupancy", 8:"cltv", 9:"dti", 10:"orig_upb", 11:"ltv",
             12:"int_rate", 16:"state", 19:"loan_id", 20:"purpose",
             21:"term", 22:"num_borrowers"}
orig = pd.read_csv("sample_orig_2018.txt", sep="|", header=None,
                   usecols=list(orig_cols.keys()))
orig = orig.rename(columns=orig_cols)

# Performance file: only 5 columns (it is large)
perf_cols = {0:"loan_id", 1:"month", 2:"upb", 3:"dq_status", 4:"loan_age"}
perf = pd.read_csv("sample_perf_2018.txt", sep="|", header=None,
                   usecols=list(perf_cols.keys()), dtype={3: str})
perf = perf.rename(columns=perf_cols)

print(orig.shape, perf.shape)
print(perf["dq_status"].value_counts().head(10))

(50000, 14) (2089132, 5)
dq_status
00    2030553
01      22925
02       7595
03       4572
04       3348
05       2868
06       2306
07       1878
08       1628
09       1455
Name: count, dtype: int64


**Reading the output:**

- The data has **50,000 loans** and **2,089,132 monthly records**, about 42 months per loan.
- Delinquency status counts months behind: `00` = current, `01` = one month behind, `02` = two months, `03` = three months, and so on. About **97% of all loan-months are current** (2,030,553 of 2,089,132), which already shows that defaults are rare in this portfolio.

## 2. Define default and build the target variable

- The standard industry definition is **90 or more days past due**. A status of `03` or higher means three or more months behind, so `dq_num >= 3` marks a default month. The code `RA` (the lender took the property after foreclosure) is also treated as default.
- Only the first **24 months of each loan** are used (`loan_age <= 24`), so every loan is observed over the same window.
- A loan counts as a default if **any** month in that window is a default (`groupby(...).max()`). This turns about two million monthly rows into one 0/1 target per loan.
- The target is merged onto the origination table by `loan_id`.
- `describe()` is used as a data-quality check before any modelling.

In [13]:
perf["dq_num"] = pd.to_numeric(perf["dq_status"], errors="coerce")
perf["is_default"] = ((perf["dq_num"] >= 3) | (perf["dq_status"] == "RA")).astype(int)

window = perf[perf["loan_age"] <= 24]
default_flag = window.groupby("loan_id")["is_default"].max().rename("default_24m")

data = orig.merge(default_flag, on="loan_id", how="left")
print(data["default_24m"].isna().sum(), "loans with no performance rows")
print("Default rate:", data["default_24m"].mean())
print(data.describe().T)

0 loans with no performance rows
Default rate: 0.0267
                  count           mean            std       min       25%  \
credit_score    50000.0     748.994880     166.683564     427.0     714.0   
first_pay_date  50000.0  201822.271020      34.285801  201802.0  201806.0   
cltv            50000.0      76.189080      17.722614       7.0      69.0   
dti             50000.0      45.701380      97.511936       1.0      29.0   
orig_upb        50000.0  231597.780000  121632.976165   19000.0  140000.0   
ltv             50000.0      75.920700      17.731111       7.0      69.0   
int_rate        50000.0       4.736412       0.476552       2.5       4.5   
term            50000.0     333.777580      61.841552      96.0     360.0   
num_borrowers   50000.0       1.468820       0.521261       1.0       1.0   
default_24m     50000.0       0.026700       0.161207       0.0       0.0   

                      50%       75%         max  
credit_score       753.00     784.0     9999.00 

**Reading the output:**

- All 50,000 loans have performance records, and the **24-month default rate is 2.67%** (about 1,335 loans). This is far lower than the roughly 22% in the credit card data, which is normal for mortgages that were credit-screened before Freddie Mac bought them.
- **Data-quality finding:** the maximum credit score is 9999 and the maximum CLTV, DTI and LTV is 999. These are Freddie Mac's "not available" placeholders, not real values. They distort the summary: DTI shows a mean of 45.7 with a standard deviation of 97.5. They are converted to missing values in the next step.
- Simplification: loans that were paid off early are counted as non-defaults.

## 3. Clean the data, split it, and fit the logistic regression

- **Placeholder codes become missing values** (9999 for credit score, 999 for CLTV, DTI and LTV), and the number of missing values in each column is printed.
- **Features:** credit score, LTV, DTI, interest rate, original balance and number of borrowers, plus loan purpose, occupancy and first-time-buyer flag. The three categorical columns are one-hot encoded (0/1 columns, with the first category dropped as the baseline).
- **CLTV is dropped** because it is almost the same as LTV (averages of 76.2 and 75.9), and keeping both would make the coefficients unstable.
- **70/30 train/test split**, stratified so that both parts keep the same default rate. `random_state=42` makes the split repeatable.
- **Pipeline:** fill missing values with the median, standardise the features, then fit the logistic regression. Because these steps sit inside a pipeline, the median and scaling are learned from the training data only, so no information leaks from the test set.
- **Metrics:** **AUC** is the chance the model ranks a random defaulter above a random non-defaulter (0.5 = coin flip, 1 = perfect). **Gini** = 2 x AUC - 1. **KS** is the largest gap between the cumulative distributions of defaulters and non-defaulters.
- The coefficients are per one-standard-deviation change in each feature, so they can be compared with each other.

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

# 1. Turn placeholder codes into missing values
data["credit_score"] = data["credit_score"].replace(9999, np.nan)
for col in ["cltv", "dti", "ltv"]:
    data[col] = data[col].replace(999, np.nan)
print(data[["credit_score", "cltv", "dti", "ltv"]].isna().sum())

# 2. Choose features (cltv is dropped because it nearly duplicates ltv)
num_features = ["credit_score", "ltv", "dti", "int_rate", "orig_upb", "num_borrowers"]
cat_features = ["purpose", "occupancy", "first_time_buyer"]
X = pd.get_dummies(data[num_features + cat_features], columns=cat_features, drop_first=True).astype(float)
y = data["default_24m"]

# 3. Split into train and test, keeping the same default rate in both
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# 4. Fill missing values, scale, then fit logistic regression
model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("logit", LogisticRegression(max_iter=1000)),
])
model.fit(X_train, y_train)

# 5. Evaluate on train and on test
def ks_stat(y_true, p):
    fpr, tpr, _ = roc_curve(y_true, p)
    return (tpr - fpr).max()

for name, Xs, ys in [("Train", X_train, y_train), ("Test", X_test, y_test)]:
    p = model.predict_proba(Xs)[:, 1]
    auc = roc_auc_score(ys, p)
    print(f"{name}: AUC={auc:.4f}  Gini={2*auc-1:.4f}  KS={ks_stat(ys, p):.4f}")

# 6. Which features matter
coefs = pd.Series(model.named_steps["logit"].coef_[0], index=X.columns).sort_values()
print(coefs)

credit_score     15
cltv              1
dti             513
ltv               1
dtype: int64
Train: AUC=0.7540  Gini=0.5079  KS=0.3841
Test: AUC=0.7693  Gini=0.5386  KS=0.4058
credit_score         -0.525805
num_borrowers        -0.229509
purpose_P            -0.099021
occupancy_P           0.017766
purpose_N             0.023401
occupancy_S           0.047169
first_time_buyer_Y    0.132038
orig_upb              0.139366
ltv                   0.239099
dti                   0.277918
int_rate              0.443055
dtype: float64


**Reading the output:**

- Missing values found and filled: 15 credit scores, 513 DTI values, and 1 each for CLTV and LTV.
- **Train AUC 0.754 (Gini 0.508, KS 0.384); test AUC 0.769 (Gini 0.539, KS 0.406).** Train and test are close, so the model is not overfitting. The test result is slightly higher by chance: the test set holds only about 400 defaults, so AUC can move by a couple of points between splits.
- **The signs make sense.** A higher credit score lowers default risk (-0.53, the strongest effect). A higher interest rate (+0.44), DTI (+0.28) and LTV (+0.24) raise it, and so does a larger original balance (+0.14). More borrowers on the loan lowers risk (-0.23).
- The interest rate partly reflects the lender's own risk pricing, which is why it ranks so high. The small purpose and occupancy coefficients are measured against the dropped baseline category and are not interpreted.

## 4. Check the probabilities (calibration) and compare with a decision tree

- **Why calibration matters:** AUC only tests the *ranking* of borrowers. Expected loss uses the probability itself, so the predicted PDs must also be accurate. A model can rank well and still be wrong on levels.
- **Decile table:** the 15,000 test loans are sorted by predicted PD and cut into 10 equal groups of 1,500. For each group the average predicted PD is compared with the actual default rate.
- **Challenger model:** a CART decision tree (`max_depth=4`, at least 200 loans per leaf) is fitted on the same training data. A challenger is a simpler or different model used to test whether the main model is the right choice.
- `export_text` prints the tree's decision rules.

In [15]:
from sklearn.tree import DecisionTreeClassifier, export_text

# Calibration: does predicted PD match actual default rate in each decile?
p_test = model.predict_proba(X_test)[:, 1]
test_df = pd.DataFrame({"actual": y_test.values, "pd": p_test})
test_df["decile"] = pd.qcut(test_df["pd"], 10, labels=False, duplicates="drop") + 1
calib = test_df.groupby("decile").agg(loans=("actual", "count"),
                                      avg_pd=("pd", "mean"),
                                      actual_rate=("actual", "mean"))
print(calib.round(4))
print("Overall predicted:", round(test_df["pd"].mean(), 4),
      "| actual:", round(test_df["actual"].mean(), 4))

# Challenger: decision tree (CART)
tree = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("tree", DecisionTreeClassifier(max_depth=4, min_samples_leaf=200, random_state=42)),
])
tree.fit(X_train, y_train)
for name, Xs, ys in [("Train", X_train, y_train), ("Test", X_test, y_test)]:
    p = tree.predict_proba(Xs)[:, 1]
    auc = roc_auc_score(ys, p)
    print(f"Tree {name}: AUC={auc:.4f}  Gini={2*auc-1:.4f}  KS={ks_stat(ys, p):.4f}")

print(export_text(tree.named_steps["tree"], feature_names=list(X.columns)))

        loans  avg_pd  actual_rate
decile                            
1        1500  0.0034       0.0033
2        1500  0.0065       0.0033
3        1500  0.0091       0.0060
4        1500  0.0121       0.0080
5        1500  0.0155       0.0153
6        1500  0.0196       0.0220
7        1500  0.0251       0.0273
8        1500  0.0333       0.0353
9        1500  0.0468       0.0493
10       1500  0.0899       0.0967
Overall predicted: 0.0261 | actual: 0.0267
Tree Train: AUC=0.7371  Gini=0.4742  KS=0.3493
Tree Test: AUC=0.7201  Gini=0.4403  KS=0.3217
|--- credit_score <= 720.50
|   |--- int_rate <= 5.19
|   |   |--- int_rate <= 4.87
|   |   |   |--- credit_score <= 639.50
|   |   |   |   |--- class: 0
|   |   |   |--- credit_score >  639.50
|   |   |   |   |--- class: 0
|   |   |--- int_rate >  4.87
|   |   |   |--- credit_score <= 646.50
|   |   |   |   |--- class: 0
|   |   |   |--- credit_score >  646.50
|   |   |   |   |--- class: 0
|   |--- int_rate >  5.19
|   |   |--- orig_upb <=

**Reading the output:**

- **Overall, the model predicts 2.61% against an actual 2.67%.** The actual default rate rises from 0.33% in the safest decile to 9.67% in the riskiest, as it should.
- Predicted and actual rates are almost identical in decile 5 (1.55% vs 1.53%). The model slightly over-predicts in deciles 2 to 4 (for example 0.65% vs 0.33% in decile 2) and slightly under-predicts in deciles 6 to 10 (8.99% vs 9.67% in decile 10).
- **Logistic regression beats the tree:** test AUC 0.769 vs 0.720, Gini 0.539 vs 0.440, KS 0.406 vs 0.322. The logistic model is also easier to explain, so it stays as the main model.
- Every leaf in the printed tree says `class 0`. That is expected, because defaults are only 2.7% of loans and no group reaches a 50% default chance. The tree is judged on ranking (AUC), not on these class labels.

| Model | Test AUC | Test Gini | Test KS |
|---|---|---|---|
| Logistic regression (main) | 0.769 | 0.539 | 0.406 |
| CART decision tree (challenger) | 0.720 | 0.440 | 0.322 |

## 5. Expected credit loss: PD x LGD x EAD

- **PD** comes from the logistic model, scored on all 50,000 loans.
- **EAD** is approximated by the original loan balance. This is a simplification, because a real CECL model uses the current outstanding balance.
- **LGD** is an **assumption**, because no recovery was estimated here. It is run at 20%, 30% and 40% to show how sensitive the result is. (Freddie Mac's data also contains actual loss fields, so estimating LGD from them is a natural next step.)
- Amounts are in US dollars, as in the data.
- `display.float_format` makes the tables print numbers with commas and two decimals.

In [16]:
pd.options.display.float_format = "{:,.2f}".format

data["pd_24m"] = model.predict_proba(X)[:, 1]
data["ead"] = data["orig_upb"]

rows = []
for lgd in [0.20, 0.30, 0.40]:
    ecl = (data["pd_24m"] * lgd * data["ead"]).sum()
    rows.append({"LGD": lgd, "Total_EAD": data["ead"].sum(),
                 "ECL": ecl, "ECL_pct_of_EAD": 100 * ecl / data["ead"].sum()})
print(pd.DataFrame(rows))

   LGD    Total_EAD            ECL  ECL_pct_of_EAD
0 0.20  11579889000  63,062,190.37            0.54
1 0.30  11579889000  94,593,285.56            0.82
2 0.40  11579889000 126,124,380.75            1.09


**Reading the output:**

- Total exposure is about **$11.58 billion** across the 50,000 loans.
- At 30% LGD the 24-month expected loss is about **$94.6 million, or 0.82% of exposure**. At 20% LGD it is $63.1 million (0.54%) and at 40% LGD it is $126.1 million (1.09%).
- Expected loss scales in direct proportion to LGD, so the LGD assumption matters as much as the PD model.
- Note: these PDs are computed on all loans, including the 70% the model was trained on, so this loss figure is an in-sample portfolio estimate. The model-quality metrics in Sections 3 and 4 come from the test set.

## 6. Stress scenario: house prices fall 30%

- **Context:** the Federal Reserve's 2026 DFAST severely adverse scenario (published 4 February 2026) has unemployment peaking at 10% and house prices falling about 30%. A CECL-style model needs a forecast, and this section shows how the loss changes under a severe one.
- **Housing channel:** if house prices fall 30%, the same loan sits against a cheaper property, so LTV rises to `LTV / (1 - 0.30)`. For example, an 80% LTV loan becomes about 114%.
- Every loan is **re-scored with the stressed LTV** using the same fitted model (no refit), and the loss is recomputed at 30% LGD and compared with the baseline.
- The results are saved to **`CECL_Summary.xlsx`** with three sheets (Scenario ECL, Calibration, LGD sensitivity), ready for a pivot table or a presentation.

In [18]:
# Stress: house prices fall 30%  ->  LTV becomes ltv / (1 - 0.30)
X_stress = X.copy()
X_stress["ltv"] = X_stress["ltv"] / (1 - 0.30)
pd_stress = model.predict_proba(X_stress)[:, 1]

lgd = 0.30
base = data["pd_24m"]
summary = pd.DataFrame({
    "Scenario": ["Baseline", "Severely adverse (house prices -30%)"],
    "Avg_PD": [base.mean(), pd_stress.mean()],
    "ECL": [(base * lgd * data["ead"]).sum(),
            (pd_stress * lgd * data["ead"]).sum()],
})
summary["ECL_pct_of_EAD"] = 100 * summary["ECL"] / data["ead"].sum()
summary["Uplift_vs_baseline_pct"] = 100 * (summary["ECL"] / summary["ECL"].iloc[0] - 1)
print(summary)

# Excel file for your CV project
with pd.ExcelWriter("CECL_Summary.xlsx") as w:
    summary.to_excel(w, sheet_name="Scenario ECL", index=False)
    calib.to_excel(w, sheet_name="Calibration")
    pd.DataFrame(rows).to_excel(w, sheet_name="LGD sensitivity", index=False)
print("Saved")

                               Scenario  Avg_PD            ECL  \
0                              Baseline    0.03  94,593,285.56   
1  Severely adverse (house prices -30%)    0.04 147,905,824.87   

   ECL_pct_of_EAD  Uplift_vs_baseline_pct  
0            0.82                    0.00  
1            1.28                   56.36  
Saved


**Reading the output:**

- Expected loss rises from **$94.6 million (0.82% of exposure) to $147.9 million (1.28%)**, an increase of **56.4%** under the severely adverse house-price shock.
- The `Avg_PD` column shows 0.03 and 0.04 only because of the two-decimal display format set in Section 5. The ECL columns carry the result.
- Only the housing channel is stressed. Unemployment, which the Fed scenario also raises, is not in this model.

## 7. Results at a glance

| Item | Result |
|---|---|
| Portfolio | 50,000 US mortgages, 2018 vintage sample |
| Default definition | 90+ days past due within 24 months of origination |
| Default rate | 2.67% |
| Logistic PD model, test set | AUC 0.769, Gini 0.539, KS 0.406 |
| Logistic PD model, train set | AUC 0.754, Gini 0.508, KS 0.384 |
| CART challenger, test set | AUC 0.720, Gini 0.440, KS 0.322 |
| Calibration, test set | Predicted 2.61% vs actual 2.67% |
| Baseline 24-month ECL (30% LGD) | $94.6M, 0.82% of $11.58B exposure |
| LGD sensitivity (20% to 40%) | 0.54% to 1.09% of exposure |
| Stress: house prices -30% | $147.9M, 1.28% of exposure, +56.4% vs baseline |

## 8. Limitations

- **24-month window, not lifetime.** CECL requires losses over the whole remaining life of a loan, including prepayments and amortisation. This notebook is a CECL-style proxy.
- **COVID overlap.** Loans started in 2018 reach their second year around early 2020, so some delinquencies reflect the pandemic and forbearance programs, not only borrower credit quality.
- **EAD is the original balance**, not the current outstanding balance.
- **LGD is assumed** (20% to 40%), not estimated from the loss data.
- **The stress is a simple proxy.** Only house prices are shocked, through LTV at origination. Unemployment is not modelled, and the model was not trained on macro variables, so this is not a calibrated macro model.
- **Expected loss is in-sample.** It is calculated on all loans, including those used to fit the model, while the quality metrics use the held-out 30%.
- **Random split on one vintage.** There is no out-of-time test (for example, train on 2017 and test on 2018).
- **Origination features only.** The model does not use how the loan behaves after it starts, and the interest rate partly reflects the lender's own risk pricing.
- **Few defaults in the test set** (about 400), so AUC and KS can move by a couple of points between splits.
- **Early payoffs** are counted as non-defaults.

## 9. Possible extensions

- Out-of-time validation: train on the 2017 vintage and test on 2018.
- Estimate LGD from the loss fields in the performance file instead of assuming it.
- Extend the horizon to lifetime loss with a time-to-default model or vintage curves, and use the current balance as EAD.
- Add macro variables such as unemployment and house price index so the scenario flows through the model directly.
- Add a gradient-boosted model as a second challenger.
- Build the Excel summary into a pivot table and a short presentation for a risk committee.

## 10. How to reproduce

1. Register on Freddie Mac's Clarity site and download the 2018 sample files (`sample_orig_2018.txt` and `sample_perf_2018.txt`) from the Single-Family Loan-Level Dataset download page.
2. Put both files in the same folder as this notebook.
3. Run the cells in order. The performance file is large, so only five columns are loaded.
4. Do not redistribute the raw data. It is licensed for internal use.